# IMT Controls — Slot Filling from Knowledge Graph
## 14 Slots · Multiple Rounds · Answerability Analysis

**Strategy:**
- Select the **first 14 slots** from the baseline
- Run **3 rounds** of LLM calls on the same 14 slots, each with a different prompt strategy
- Compare results across rounds to determine which slots are **consistently answerable**, **inconsistent**, or **systematically unanswerable**
- Draw conclusions on **which question styles** the graph can/cannot answer

**The 3 rounds:**
| Round | Strategy |
|-------|----------|
| Round 1 | Direct filling — ask the LLM to fill slots straight from the graph |
| Round 2 | Guided filling — give the LLM hints about where to look in the graph (node types) |
| Round 3 | Conservative filling — instruct the LLM to only answer if evidence is explicit and unambiguous |


## Bloc 1 — Imports & Configuration

In [ ]:
!pip install pandas python-dotenv langchain-nvidia-ai-endpoints langchain-core -q

import os
import json
import math
import pandas as pd
from collections import Counter
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA

load_dotenv()

print("Imports OK")
print(f"NVIDIA_API_KEY : {'OK' if os.getenv('NVIDIA_API_KEY') else 'MISSING'}")


## Bloc 2 — LLM Initialization

In [ ]:
llm = ChatNVIDIA(
    model="mistralai/mistral-large-3-675b-instruct-2512",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0,
    max_completion_tokens=8000,
)
print("LLM ready")


## Bloc 3 — Load Knowledge Graph

In [ ]:
GRAPH_JSON_PATH = "SecuraOps High_graph_4pass.json"   # <-- update if needed

with open(GRAPH_JSON_PATH, "r", encoding="utf-8") as f:
    graph = json.load(f)

entities  = graph.get("entities",      [])
relations = graph.get("relationships", [])

print(f"Graph loaded: {len(entities)} entities, {len(relations)} relationships")

label_counts = Counter(e["label"] for e in entities)
print("\nNode types:")
for label, count in sorted(label_counts.items()):
    print(f"  {label:<22}: {count}")


## Bloc 4 — Load Baseline & Select First 14 Slots

In [ ]:
BASELINE_CSV = "Baseline_pour_IMT_Controls_Slots_IMT_.csv"

df_all = pd.read_csv(BASELINE_CSV, sep=";", encoding="utf-8-sig")
desc_col = [c for c in df_all.columns if "Description" in c][0]
df_all.rename(columns={desc_col: "description"}, inplace=True)

SLOT_COUNT = 14
df_slots = df_all.head(SLOT_COUNT).reset_index(drop=True)

print(f"Total slots in baseline : {len(df_all)}")
print(f"Slots selected          : {len(df_slots)}")
print()
print(df_slots[["Control_ID", "Control_Name", "Slot_Key", "Slot_Type"]].to_string(index=False))


## Bloc 5 — Build Graph Summary

In [ ]:
def build_graph_summary(entities: list, relationships: list) -> str:
    lines_e = []
    for e in entities:
        props = ", ".join(f'{k}="{v}"' for k, v in e.get("properties", {}).items())
        lines_e.append(f"  [{e['label']}]  {props}")

    lines_r = []
    for r in relationships:
        from_val = list(r.get("from_key", {}).values())
        to_val   = list(r.get("to_key",   {}).values())
        lines_r.append(
            f"  ({r['from_label']}:{from_val[0] if from_val else '?'})"
            f" -[{r['rel_type']}]->"
            f" ({r['to_label']}:{to_val[0] if to_val else '?'})"
        )

    return (
        "=== ENTITIES ===\n" + "\n".join(lines_e) +
        "\n\n=== RELATIONSHIPS ===\n" + "\n".join(lines_r)
    )

GRAPH_SUMMARY = build_graph_summary(entities, relations)
print(f"Graph summary: {len(GRAPH_SUMMARY):,} chars (~{len(GRAPH_SUMMARY)//4:,} tokens estimated)")


## Bloc 6 — Slot Formatter (shared across rounds)

In [ ]:
def format_slot(row) -> str:
    """Format a single slot into an instruction block for the LLM."""
    stype = str(row.get("Slot_Type", "bool"))
    enum  = row.get("Enum_Values")
    try:
        if math.isnan(float(enum)): enum = None
    except (TypeError, ValueError): pass

    if stype == "bool":
        type_hint = "bool  →  true  or  false"
    elif stype == "enum" and enum:
        type_hint = f"enum  →  one of:  {enum}"
    elif stype == "list[string]":
        type_hint = 'list[string]  →  JSON array, e.g. ["AES-256"]' 
    else:
        type_hint = "bool  →  true  or  false"

    return (
        f"slot_key   : \"{row['Slot_Key']}\""
        f"\ncontrol    : {row['Control_ID']} — {row['Control_Name']}"
        f"\ntype       : {type_hint}"
        f"\ninstruction: {row['description']}"
    )

SLOT_BLOCKS = "\n\n".join(format_slot(row) for _, row in df_slots.iterrows())
print("Slot formatter ready")
print(f"Slot block preview:\n{SLOT_BLOCKS[:400]}...")


## Bloc 7 — Define the 3 Round Prompts

Each round uses the **same graph** and **same 14 slots** but a **different system prompt strategy**.
This lets us observe how much the prompt strategy affects answerability.


In [ ]:
OUTPUT_FORMAT = """
Return ONLY a valid JSON array (no markdown, no explanation):
[
  {
    "slot_key"     : "<key>",
    "control_id"   : "<Control_ID>",
    "value"        : <true|false|"enum_value"|["item"]|null>,
    "answerability": "answerable" | "ambiguous" | "unanswerable",
    "evidence"     : "<one-sentence justification or 'Not found in graph'>"
  },
  ...
]"""

# ─────────────────────────────────────────────────────────────────────────────
ROUND_PROMPTS = {

    1: {
        "label": "Round 1 — Direct filling",
        "description": "Ask the LLM to fill slots directly from the graph with no extra guidance.",
        "system": (
            "You are a cybersecurity analyst filling a security-assessment baseline "
            "from a knowledge graph extracted from supplier documents.\n\n"
            "Rules:\n"
            "- Answer ONLY from what is present in the knowledge graph.\n"
            "- For bool: return true or false.\n"
            "- For enum: return exactly one allowed value.\n"
            "- For list[string]: return a JSON array of strings.\n"
            "- If the graph does not contain enough information → value: null, answerability: unanswerable.\n"
            "- If information is partial → answerability: ambiguous.\n"
            "- If clearly supported → answerability: answerable.\n"
            "- Always include a short evidence field (≤1 sentence)."
            + OUTPUT_FORMAT
        ),
    },

    2: {
        "label": "Round 2 — Guided filling (node-type hints)",
        "description": (
            "Same task but the LLM is told which node types to look for, "
            "reducing hallucination and improving recall."
        ),
        "system": (
            "You are a cybersecurity analyst filling a security-assessment baseline "
            "from a knowledge graph extracted from supplier documents.\n\n"
            "The graph contains these node types: Supplier, Agreement, Clause, Control, "
            "Algorithm, Protocol, Asset, Role, Technology, Claim.\n\n"
            "For each slot:\n"
            "  - ACH slots   → look for Supplier, Agreement, Clause nodes.\n"
            "  - DATA slots  → look for Algorithm, Protocol, Asset nodes.\n\n"
            "Rules:\n"
            "- Answer ONLY from what is present in the knowledge graph.\n"
            "- For bool: return true or false.\n"
            "- For enum: return exactly one allowed value.\n"
            "- For list[string]: return a JSON array of strings.\n"
            "- If the graph does not contain enough information → value: null, answerability: unanswerable.\n"
            "- If information is partial → answerability: ambiguous.\n"
            "- If clearly supported → answerability: answerable.\n"
            "- Always include a short evidence field (≤1 sentence)."
            + OUTPUT_FORMAT
        ),
    },

    3: {
        "label": "Round 3 — Conservative filling (strict evidence)",
        "description": (
            "LLM instructed to answer ONLY when evidence is explicit and unambiguous. "
            "Any doubt → unanswerable. This round surfaces the hard structural limits."
        ),
        "system": (
            "You are a strict cybersecurity auditor filling a security-assessment baseline. "
            "You must be highly conservative.\n\n"
            "Rules:\n"
            "- Answer ONLY if there is an EXPLICIT, UNAMBIGUOUS node or property in the graph.\n"
            "- If you have ANY doubt → value: null, answerability: unanswerable.\n"
            "- Never infer, assume, or extrapolate beyond what is literally written in the graph.\n"
            "- For bool: return true or false.\n"
            "- For enum: return exactly one allowed value.\n"
            "- For list[string]: return a JSON array of strings.\n"
            "- answerability must be 'answerable' only when evidence is direct and explicit.\n"
            "- Always include a short evidence field (≤1 sentence)."
            + OUTPUT_FORMAT
        ),
    },
}

print("Round prompts defined:")
for rnd, cfg in ROUND_PROMPTS.items():
    print(f"  Round {rnd}: {cfg['label']}")
    print(f"           {cfg['description']}")


## Bloc 8 — LLM Call & JSON Parser

In [ ]:
def call_llm(system_prompt: str, user_prompt: str) -> str:
    messages = [("system", system_prompt), ("human", user_prompt)]
    raw = llm.invoke(messages).content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"): raw = raw[4:]
        raw = raw.strip()
    return raw


def parse_results(raw: str, df_slots: pd.DataFrame, round_num: int) -> list:
    try:
        results = json.loads(raw)
        if not isinstance(results, list): raise ValueError("Not a list")
    except Exception as e:
        print(f"  [PARSE ERROR Round {round_num}] {e}")
        results = []

    result_map = {r.get("slot_key"): r for r in results}
    rows = []
    for _, slot in df_slots.iterrows():
        key = slot["Slot_Key"]
        r   = result_map.get(key, {})
        rows.append({
            "round"        : round_num,
            "control_id"   : slot["Control_ID"],
            "control_name" : slot["Control_Name"],
            "slot_key"     : key,
            "slot_type"    : slot["Slot_Type"],
            "value"        : r.get("value"),
            "answerability": r.get("answerability", "error" if not r else "unanswerable"),
            "evidence"     : r.get("evidence", "Not returned by model"),
        })
    return rows


print("LLM helpers ready")


## Bloc 9 — ▶️ Run All 3 Rounds

In [ ]:
all_rows = []

for round_num, cfg in ROUND_PROMPTS.items():
    print(f"\n{'─'*60}")
    print(f"{cfg['label']}")
    print(f"Strategy: {cfg['description']}")
    print(f"{'─'*60}")

    user_prompt = (
        f"KNOWLEDGE GRAPH:\n{GRAPH_SUMMARY}\n\n"
        f"SLOTS TO FILL ({SLOT_COUNT} slots):\n\n{SLOT_BLOCKS}\n\n"
        "Fill all 14 slots using only the knowledge graph above. Return only the JSON array."
    )

    try:
        raw = call_llm(cfg["system"], user_prompt)
        rows = parse_results(raw, df_slots, round_num)
        all_rows.extend(rows)

        for r in rows:
            icon = {"answerable": "✅", "ambiguous": "⚠️",
                    "unanswerable": "❌", "error": "💥"}.get(r["answerability"], "?")
            print(f"  {icon} {r['slot_key']:<40} = {str(r['value']):<12} [{r['answerability']}]")

    except Exception as e:
        print(f"  [ERROR] {e}")

print(f"\n{'='*60}")
print(f"All rounds complete. Total records: {len(all_rows)}")


## Bloc 10 — Build Results DataFrame

In [ ]:
df_all_rounds = pd.DataFrame(all_rows)

print("Results shape:", df_all_rounds.shape)
print()

# Summary per round
for rnd in [1, 2, 3]:
    df_r = df_all_rounds[df_all_rounds["round"] == rnd]
    counts = df_r["answerability"].value_counts()
    print(f"Round {rnd}: ✅ {counts.get('answerable',0)}  ⚠️ {counts.get('ambiguous',0)}  ❌ {counts.get('unanswerable',0)}")

df_all_rounds.head(14)


## Bloc 11 — Cross-Round Comparison per Slot

In [ ]:
# Pivot: one row per slot, one column per round
pivot = df_all_rounds.pivot_table(
    index=["control_id", "slot_key"],
    columns="round",
    values="answerability",
    aggfunc="first"
).reset_index()
pivot.columns = ["control_id", "slot_key", "round_1", "round_2", "round_3"]

# Classify consistency
def classify_consistency(row):
    vals = [row["round_1"], row["round_2"], row["round_3"]]
    if all(v == "answerable"   for v in vals): return "stable_answerable"
    if all(v == "unanswerable" for v in vals): return "stable_unanswerable"
    if "answerable" in vals and "unanswerable" in vals: return "inconsistent"
    if all(v == "ambiguous"    for v in vals): return "stable_ambiguous"
    return "mixed"

pivot["consistency"] = pivot.apply(classify_consistency, axis=1)

print("Cross-round slot comparison:")
print(pivot.to_string(index=False))
print()
print("Consistency summary:")
print(pivot["consistency"].value_counts().to_string())


## Bloc 12 — Answerability Analysis

**Which question styles can the graph answer? Which can it not — and why?**

This is informed by the 3-round experiment:
- **stable_answerable** → the graph reliably answers this type of question
- **stable_unanswerable** → structural gap in the graph for this question type
- **inconsistent** → the graph has partial information; prompt strategy affects the result


In [ ]:
print("=" * 65)
print("WHAT THE GRAPH CAN ANSWER  (stable_answerable slots)")
print("=" * 65)

stable_ans = pivot[pivot["consistency"] == "stable_answerable"]["slot_key"].tolist()
print(f"Slots: {stable_ans}")
print("""
These slots share a common pattern — they are EXISTENCE questions:
  "Does a security clause exist?"       → ACH-01, ACH-02
  "Is encryption documented?"           → DATA-01, DATA-02
  "Are named algorithms present?"       → DATA-02.algorithms

Why the graph handles them well:
  → The graph has dedicated node types (Clause, Algorithm, Protocol)
    with named properties directly extracted from the documents.
  → Existence is confirmed by the presence of a node — straightforward
    graph traversal, no inference needed.
  → All 3 rounds agree → the answer does not depend on prompt strategy.
""")

print("=" * 65)
print("WHAT THE GRAPH CANNOT ANSWER  (stable_unanswerable slots)")
print("=" * 65)

stable_unans = pivot[pivot["consistency"] == "stable_unanswerable"]["slot_key"].tolist()
print(f"Slots: {stable_unans}")
print("""
These slots fail in all 3 rounds regardless of prompt strategy.
They reveal STRUCTURAL GAPS in the graph:

  1. NEGATIVE EVIDENCE questions
     e.g. "Is NDA the ONLY clause used?"  → ACH-02.nda_only
     Why: The graph captures what IS present, not what is ABSENT.
          Confirming exclusivity requires reading the full document.

  2. INFRASTRUCTURE-LEVEL CONFIGS not in policy documents
     e.g. "Is HSTS enforced?", "Is PFS enforced?" → DATA-01.hsts / pfs
     Why: These are web-server settings. The source documents are
          governance/policy texts — they don't describe server configs.

  3. OPERATIONAL CADENCE
     e.g. "Is key rotation performed regularly?" → DATA-02.key_rotation
     Why: The graph captures WHAT is implemented, not HOW OFTEN.
          Frequency/scheduling is rarely a named node in the schema.
""")

print("=" * 65)
print("INCONSISTENT SLOTS  (prompt-sensitive)")
print("=" * 65)

inconsistent = pivot[pivot["consistency"] == "inconsistent"]["slot_key"].tolist()
print(f"Slots: {inconsistent}")
print("""
These slots changed answer depending on the round (prompt strategy).
They reveal PARTIAL INFORMATION in the graph:

  → Round 1 (direct) may hallucinate an answer.
  → Round 2 (guided) finds relevant nodes with better targeting.
  → Round 3 (conservative) marks them unanswerable due to ambiguity.

Conclusion: The graph contains *hints* but not *explicit* evidence.
These slots need richer extraction (more passes, finer-grained nodes).
""")


## Bloc 13 — Visual Summary

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("14-Slot Filling — Multi-Round Answerability Analysis", fontsize=13, fontweight="bold")

# ── Plot 1: Answerability per round (stacked bar) ────────────────────────────
ax1 = axes[0]
categories = ["answerable", "ambiguous", "unanswerable"]
colors     = ["#4caf50", "#ff9800", "#f44336"]
rounds     = [1, 2, 3]
labels     = [ROUND_PROMPTS[r]["label"].split("—")[1].strip() for r in rounds]
bottoms    = [0] * 3

for cat, color in zip(categories, colors):
    vals = [
        len(df_all_rounds[(df_all_rounds["round"] == r) & (df_all_rounds["answerability"] == cat)])
        for r in rounds
    ]
    ax1.bar(labels, vals, bottom=bottoms, color=color, label=cat.capitalize())
    bottoms = [b + v for b, v in zip(bottoms, vals)]

ax1.set_title("Answerability per Round")
ax1.set_ylabel("Number of slots")
ax1.set_ylim(0, 14)
ax1.legend()
ax1.tick_params(axis="x", rotation=10)

# ── Plot 2: Consistency classification ───────────────────────────────────────
ax2 = axes[1]
consist_counts = pivot["consistency"].value_counts()
consist_colors = {
    "stable_answerable"  : "#4caf50",
    "stable_unanswerable": "#f44336",
    "inconsistent"       : "#ff9800",
    "stable_ambiguous"   : "#9e9e9e",
    "mixed"              : "#607d8b",
}
bar_colors = [consist_colors.get(k, "#aaa") for k in consist_counts.index]
ax2.bar(consist_counts.index, consist_counts.values, color=bar_colors)
ax2.set_title("Slot Consistency Across 3 Rounds")
ax2.set_ylabel("Number of slots")
ax2.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("14slots_multiround_analysis.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: 14slots_multiround_analysis.png")


## Bloc 14 — Export Results

In [ ]:
# All rounds CSV
df_all_rounds.to_csv("14slots_all_rounds.csv", index=False, encoding="utf-8-sig")
print("Saved: 14slots_all_rounds.csv")

# Cross-round comparison CSV
pivot.to_csv("14slots_consistency.csv", index=False, encoding="utf-8-sig")
print("Saved: 14slots_consistency.csv")

# Final display
print("\nFinal cross-round comparison table:")
df_all_rounds[["round", "control_id", "slot_key", "value", "answerability", "evidence"]]
